# TUM Depth Camera Semantic Navigation Demo

Demo-ноутбук для Google Colab: строит семантическую 3D карту на данных TUM RGB-D, сегментирует объекты масками (SAM), добавляет подписи через CLIP и показывает поиск объекта по тексту.


## 1. Настройка окружения и параметров
Запустите ячейку ниже в Colab: установятся зависимости и зададутся основные параметры.

In [ ]:
# Установка зависимостей (Colab)
%pip -q install open3d==0.17.0 opencv-python-headless==4.10.0.84 "git+https://github.com/openai/CLIP.git" segment-anything==1.0 numpy matplotlib tqdm

import os
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple, Dict

import cv2
import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d
import torch
import clip
from PIL import Image
from tqdm import tqdm
from segment_anything import sam_model_registry, SamPredictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'

@dataclass
class Config:
    sequence_url: str = 'https://cvg.cit.tum.de/rgbd/dataset/freiburg1/rgbd_dataset_freiburg1_desk.tgz'
    dataset_root: Path = Path('tum_rgbd')
    max_frames: int = 80  # используйте 30-80 кадров для быстрой демонстрации
    frame_stride: int = 2  # шаг по кадрам, чтобы ускорить работу
    depth_scale: float = 5000.0  # TUM хранит глубину в миллиметрах
    mask_score_thresh: float = 0.86
    sam_checkpoint_url: str = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'
    sam_model: str = 'vit_b'
    text_queries: List[str] = None

    def __post_init__(self):
        if self.text_queries is None:
            self.text_queries = ['стул', 'стол', 'монитор', 'человек', 'кружка']

config = Config()
config.dataset_root.mkdir(parents=True, exist_ok=True)
print(f'Используемое устройство: {device}')


## 2. Загрузка TUM RGB-D, подготовка пар кадров и построение 3D карты
Ячейка скачивает последовательность, парсит `rgb.txt`, `depth.txt`, `groundtruth.txt`, сопоставляет кадры по времени и строит глобальную карту точек в Open3D.

In [ ]:
import tarfile
import urllib.request

seq_dir = config.dataset_root / 'fr1_desk'
tar_path = config.dataset_root / 'fr1_desk.tgz'

if not seq_dir.exists():
    if not tar_path.exists():
        print('Скачиваем последовательность fr1/desk (~1.1GB)...')
        urllib.request.urlretrieve(config.sequence_url, tar_path)
    with tarfile.open(tar_path) as tar:
        tar.extractall(config.dataset_root)
    print('Готово!')

rgb_txt = seq_dir / 'rgb.txt'
depth_txt = seq_dir / 'depth.txt'
gt_txt = seq_dir / 'groundtruth.txt'

def load_timestamps(file_path: Path) -> List[Tuple[float, str]]:
    entries = []
    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith('#') or len(line.strip()) == 0:
                continue
            parts = line.strip().split()
            entries.append((float(parts[0]), parts[1]))
    return entries

def load_poses(gt_file: Path) -> Dict[float, np.ndarray]:
    poses = {}
    with open(gt_file, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            t, tx, ty, tz, qx, qy, qz, qw = line.strip().split()
            t = float(t)
            trans = np.array([float(tx), float(ty), float(tz)])
            quat = np.array([float(qw), float(qx), float(qy), float(qz)])  # w, x, y, z
            R = o3d.geometry.get_rotation_matrix_from_quaternion(quat)
            pose = np.eye(4)
            pose[:3, :3] = R
            pose[:3, 3] = trans
            poses[t] = pose
    return poses

def associate(rgb_entries, depth_entries, max_diff=0.02):
    pairs = []
    depth_times = [d[0] for d in depth_entries]
    j = 0
    for t_rgb, path_rgb in rgb_entries:
        while j < len(depth_entries) - 1 and abs(depth_times[j+1] - t_rgb) < abs(depth_times[j] - t_rgb):
            j += 1
        t_depth, path_depth = depth_entries[j]
        if abs(t_depth - t_rgb) <= max_diff:
            pairs.append((t_rgb, path_rgb, t_depth, path_depth))
    return pairs

rgb_entries = load_timestamps(rgb_txt)
depth_entries = load_timestamps(depth_txt)
poses = load_poses(gt_txt)
pairs = associate(rgb_entries, depth_entries)
print(f'Найдено пар RGB/Depth: {len(pairs)}')

intrinsic = o3d.camera.PinholeCameraIntrinsic(o3d.camera.PinholeCameraIntrinsicParameters.PrimeSenseDefault)
map_pcd = o3d.geometry.PointCloud()

selected_pairs = pairs[: config.max_frames : config.frame_stride]
for (t_rgb, path_rgb, t_depth, path_depth) in tqdm(selected_pairs, desc='Интеграция точек'):
    rgb = o3d.io.read_image(str(seq_dir / path_rgb))
    depth_raw = o3d.io.read_image(str(seq_dir / path_depth))
    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        rgb, depth_raw, depth_scale=config.depth_scale, depth_trunc=5.0, convert_rgb_to_intensity=False
    )
    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic)
    pose = poses.get(t_rgb)
    if pose is None:
        continue
    pcd.transform(pose)
    map_pcd += pcd

map_pcd = map_pcd.voxel_down_sample(voxel_size=0.01)
o3d.io.write_point_cloud('tum_map.ply', map_pcd)
print('Карта сохранена: tum_map.ply')


## 3. Выделение объектов масками (SAM)
Берём один из кадров, предсказываем маски Segment Anything, фильтруем по уверенности и проецируем выбранный объект в 3D.

In [ ]:
def load_sam(checkpoint_path: Path):
    sam = sam_model_registry[config.sam_model](checkpoint=checkpoint_path)
    sam.to(device)
    return SamPredictor(sam)

sam_ckpt = config.dataset_root / Path(config.sam_checkpoint_url).name
if not sam_ckpt.exists():
    print('Скачиваем чекпойнт SAM...')
    urllib.request.urlretrieve(config.sam_checkpoint_url, sam_ckpt)
predictor = load_sam(sam_ckpt)

sample_idx = min(10, len(selected_pairs)-1)
_, rgb_path, _, depth_path = selected_pairs[sample_idx]
rgb_bgr = cv2.cvtColor(cv2.imread(str(seq_dir / rgb_path)), cv2.COLOR_BGR2RGB)
predictor.set_image(rgb_bgr)
masks, scores, _ = predictor.predict(multimask_output=True)

good_masks = []
for m, s in zip(masks, scores):
    if s >= config.mask_score_thresh:
        good_masks.append((m, s))

print(f'Масок после фильтрации: {len(good_masks)}')

plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.imshow(rgb_bgr); plt.axis('off'); plt.title('RGB кадр')
if good_masks:
    overlay = rgb_bgr.copy()
    for m,_ in good_masks:
        overlay[m] = overlay[m]*0.5 + np.array([0,255,0])*0.5
    plt.subplot(1,3,2); plt.imshow(overlay); plt.axis('off'); plt.title('SAM маски')
plt.subplot(1,3,3); plt.imshow(good_masks[0][0] if good_masks else masks[0]); plt.axis('off'); plt.title('Пример маски')
plt.show()

depth_img = np.asarray(o3d.io.read_image(str(seq_dir / depth_path)), dtype=np.float32) / config.depth_scale
selected_mask = good_masks[0][0] if good_masks else masks[0]
ys, xs = np.where(selected_mask)

# 3D точки выбранного объекта
u = xs.astype(np.float32)
v = ys.astype(np.float32)
z = depth_img[ys, xs]
fx, fy = intrinsic.get_focal_length()
cx, cy = intrinsic.get_principal_point()
x = (u - cx) * z / fx
y = (v - cy) * z / fy
object_points = np.stack([x, y, z], axis=1)
print(f'Точек в маске: {object_points.shape[0]}')


## 4. Добавление семантики с CLIP
Функция ниже вычисляет эмбеддинги для текстовых запросов и масок, чтобы сравнивать объекты по тексту.

In [ ]:
clip_model, preprocess = clip.load('ViT-B/32', device=device)
text_tokens = clip.tokenize(config.text_queries).to(device)
text_features = clip_model.encode_text(text_tokens)
text_features /= text_features.norm(dim=-1, keepdim=True)

def crop_mask(image: np.ndarray, mask: np.ndarray) -> Image.Image:
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return Image.fromarray(image)
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    crop = image[y0:y1+1, x0:x1+1]
    return Image.fromarray(crop)

mask_image = crop_mask(rgb_bgr, selected_mask)
with torch.no_grad():
    image_tensor = preprocess(mask_image).unsqueeze(0).to(device)
    image_features = clip_model.encode_image(image_tensor)
    image_features /= image_features.norm(dim=-1, keepdim=True)

probs = (100.0 * image_features @ text_features.T).softmax(dim=-1).squeeze(0)
for label, p in zip(config.text_queries, probs):
    print(f'{label:10s}: {float(p)*100:.1f}%')

best_idx = int(probs.argmax())
best_label = config.text_queries[best_idx]
print(f'Маска классифицирована как: {best_label}')


## 5. Пример поиска объекта на карте
Текстовый запрос -> косинусная близость к эмбеддингам объектов (масок). Можно расширить на все маски и объединять точки с одинаковой меткой в облаке.

In [ ]:
def find_best_object(mask_list: List[np.ndarray], rgb_image: np.ndarray, query: str):
    tokens = clip.tokenize([query]).to(device)
    with torch.no_grad():
        query_feat = clip_model.encode_text(tokens)
        query_feat /= query_feat.norm(dim=-1, keepdim=True)
    best_score, best_id = -1, -1
    for idx, mask in enumerate(mask_list):
        crop = crop_mask(rgb_image, mask)
        with torch.no_grad():
            img_feat = clip_model.encode_image(preprocess(crop).unsqueeze(0).to(device))
            img_feat /= img_feat.norm(dim=-1, keepdim=True)
        score = float((img_feat @ query_feat.T).squeeze())
        if score > best_score:
            best_score, best_id = score, idx
    return best_id, best_score

all_masks = [m for m,_ in good_masks] if good_masks else [masks[0]]
query = 'кружка'
best_id, score = find_best_object(all_masks, rgb_bgr, query)
print(f'Лучший объект для запроса "{query}": маска {best_id} (score={score:.3f})')

search_overlay = rgb_bgr.copy()
search_overlay[all_masks[best_id]] = search_overlay[all_masks[best_id]]*0.5 + np.array([255,0,0])*0.5
plt.figure(figsize=(8,4))
plt.subplot(1,2,1); plt.imshow(rgb_bgr); plt.axis('off'); plt.title('RGB')
plt.subplot(1,2,2); plt.imshow(search_overlay); plt.axis('off'); plt.title(f'Результат поиска: {query}')
plt.show()

# Пример сохранения точек найденного объекта
object_mask = all_masks[best_id]
ys, xs = np.where(object_mask)
z = depth_img[ys, xs]
x = (xs - cx) * z / fx
y = (ys - cy) * z / fy
pts = np.stack([x, y, z], axis=1)
obj_pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(pts))
o3d.io.write_point_cloud(f'query_{query}.ply', obj_pcd)
print('Сохранены точки найденного объекта для визуализации в MeshLab/CloudCompare.')


---
Ноутбук можно расширить: 
* добавлять все маски и подписи в глобальную карту; 
* использовать более компактные модели SAM/CLIP; 
* сохранять KD-дерево с позами объектов для быстрого поиска ближайшего по направлению.